# Lab 5: Hodgkin–Huxley Simulation of Electrical Stimulation

Until now, we have mainly **read from** the nervous system by recording signals, extracting features, and decoding neural state or behavior. In this lab, we turn to **writing to** the nervous system by simulating how applied current changes a neuron's membrane voltage.

We will use a single-compartment Hodgkin–Huxley (HH) model to study how stimulation amplitude, pulse width, and frequency affect spiking. We will then construct a strength–duration curve, estimate rheobase and chronaxie, compare monophasic and charge-balanced biphasic stimulation, and evaluate an electrode design using a tissue-safety criterion from lecture.

### Learning goals

By the end of the lab, you should be able to:

- Implement the HH membrane equation and explain how applied current interacts with ionic currents.
- Predict how stimulation amplitude, pulse width, and frequency affect activation and recovery.
- Construct a strength–duration curve and interpret rheobase and chronaxie.
- Explain how pulse width and interphase gap affect monophasic and biphasic activation thresholds.
- Calculate electrode charge density and explain the limitations of simplified safety criteria.


## 1. Setup

This lab uses only NumPy and Matplotlib, which were installed for the previous labs. No additional packages are required. Before opening the notebook, activate the same course environment used for the earlier labs:

```bash
conda activate ee194-21
```

The simulation uses a small, documented NumPy implementation rather than a specialized neural simulator. This lets us see exactly where the stimulation current enters the model, and you will write that part yourself.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import ScalarFormatter

np.set_printoptions(precision=3, suppress=True)

## 2. A Hodgkin–Huxley neuron receiving stimulation

Recall from lecture that the membrane behaves like a capacitor: current crossing the membrane charges it and changes the membrane voltage ($\Delta V_m = i_m \Delta t / C_m$). The Hodgkin–Huxley model adds voltage-dependent sodium and potassium conductances and a leak conductance to that capacitor:

$$
C_m\frac{dV}{dt}
=
I_{\mathrm{stim}}
-I_{\mathrm{Na}}
-I_{\mathrm{K}}
-I_{\mathrm{leak}}.
$$

By convention in this model, a positive $I_{\mathrm{stim}}$ is positive current flowing into the cell (equivalently, negative current flowing out), which depolarizes the membrane. The ionic currents $I_{\mathrm{Na}}$, $I_{\mathrm{K}}$, and $I_{\mathrm{leak}}$ are instead defined as positive when positive charge flows out of the cell, which is why they are subtracted in the membrane equation.

Each ionic current is a conductance ($1/\text{resistance}$) multiplied by the difference between the membrane voltage and that ion's reversal potential. The reversal potential is the membrane voltage at which there is no net current of that ion through its channels. For a given conductance, the farther the membrane voltage is from the ion's reversal potential, the larger the transmembrane current carried by that ion:

$$
I_{\mathrm{Na}} = \bar g_{\mathrm{Na}}\, m^3 h\,(V - E_{\mathrm{Na}}),
\qquad
I_{\mathrm{K}} = \bar g_{\mathrm{K}}\, n^4\,(V - E_{\mathrm{K}}),
\qquad
I_{\mathrm{leak}} = \bar g_{\mathrm{leak}}\,(V - E_{\mathrm{leak}}).
$$

In this model, $E_{\mathrm{Na}}=+50$ mV, $E_{\mathrm{K}}=-77$ mV, and $E_{\mathrm{leak}}=-54.4$ mV. Near the resting voltage ($V\approx-65$ mV), $V-E_{\mathrm{Na}}$ is negative, so opening sodium channels produces an inward current that drives $V$ upward toward $+50$ mV. In contrast, $V-E_{\mathrm{K}}$ is positive, so opening potassium channels produces an outward current that drives $V$ downward toward $-77$ mV. Thus, sodium current primarily produces the rising phase of the spike, whereas the delayed potassium current helps repolarize the membrane.

The gating variables $m$, $h$, and $n$ are mathematical descriptions of channel gates, not ion concentrations. Each varies between 0 and 1: a value near 0 means that few gates of that type are in a state that permits current, whereas a value near 1 means that most are. Sodium conductance depends on $m^3h$ because the model represents three sodium activation gates ($m$) and one sodium availability gate ($h$); all must be permissive for the channel to conduct. Potassium conductance similarly depends on four activation gates and is proportional to $n^4$.

The gates do not switch instantly when voltage changes. Their opening and closing rates depend on the membrane voltage $V_m$, so each gate's value reflects both the current voltage and the recent voltage history. Changes in $V_m$ alter the gates, which alter the ionic currents and then feed back onto $V_m$:

- $m$: **sodium activation**. Depolarization rapidly increases $m$, opening more sodium channels. Repolarization decreases $m$.
- $h$: **sodium availability/inactivation**. At rest, $h$ is relatively high and sodium channels are available. Depolarization decreases $h$ more slowly, inactivating sodium channels; $h$ recovers as the membrane repolarizes.
- $n$: **potassium activation**. Depolarization increases $n$ more slowly than $m$, opening potassium channels. After repolarization, $n$ gradually decreases.

These different time courses produce an action potential. An initial depolarization rapidly increases $m$, increasing inward sodium current and causing still more depolarization. This positive-feedback loop produces the rapid, all-or-none rise of the spike. Meanwhile, $h$ slowly decreases and $n$ slowly increases, reducing sodium current and increasing outward potassium current. Together, these slower changes repolarize the membrane. Immediately afterward, $h$ remains low and $n$ remains high for a short time, so less sodium current is available and more potassium current opposes depolarization. The neuron is therefore temporarily harder to excite.

### Connecting the model to extracellular stimulation

In lecture, **cathodic** (negative) current at the electrode was the excitatory phase. In this model, a **positive** $I_{\mathrm{stim}}$ is excitatory. These statements are consistent. Recall that the membrane current along an axon is proportional to the second spatial derivative of the extracellular voltage $V_e$. Under a cathode, $V_e$ has a sharp negative minimum, so the membrane closest to the electrode is **depolarized** (and the flanking membrane is slightly hyperpolarized). Our single compartment represents one patch of membrane, so we apply that net effect directly: positive $I_{\mathrm{stim}}$ depolarizes the membrane, like a cathodic phase acting on the membrane nearest the electrode, and negative $I_{\mathrm{stim}}$ hyperpolarizes it, like an anodic phase. In this lab we say *depolarizing-first* where a clinical waveform would say *cathodic-first*.

### An important modeling limitation

Here, $I_{\mathrm{stim}}$ is an **effective current density applied directly to the modeled membrane**. Real extracellular stimulation creates a spatial electric field that polarizes different parts of axons and other neural elements differently. Therefore, this model is useful for studying membrane excitability and waveform tradeoffs, but it does not model electrode geometry, the extracellular field, axon orientation, or recruitment of a neural population.

### 2.1 Implement the membrane equation

The cell below contains the full HH simulator. Everything is provided **except the membrane equation** inside the time loop of `simulate_hh`. Complete the four lines marked `## student code`:

1. `i_na`, `i_k`, and `i_leak`: the three ionic currents, using the equations above. In the code, $\bar g_{\mathrm{Na}}$ and $E_{\mathrm{Na}}$ are `g_na` and `e_na` (similarly for K and leak), the current gate values are `m`, `h`, and `n`, and the current voltage is `v`. Use `**` for powers.
2. `dv_dt`: rearrange the membrane equation to solve for $dV/dt$. The applied current at the current time step is `stimulus[sample]` and the membrane capacitance is `capacitance`.

Read the docstrings as well. The optional `return_gates` argument will be used later to inspect channel recovery.


In [ ]:
# TODO: Complete the four marked lines in the Hodgkin–Huxley membrane equation.
# Hodgkin–Huxley model code. Complete only the lines marked "## student code".
# Units: voltage in mV, time in ms, current density in µA/cm², capacitance in µF/cm²,
# and conductance in mS/cm².

def _vtrap(x, scale):
    """Evaluate x / (exp(x/scale) - 1) without a numerical 0/0."""
    x = np.asarray(x, dtype=float)
    z = x / scale
    result = np.empty_like(z)
    small = np.abs(z) < 1e-6
    result[small] = scale * (1 - z[small] / 2 + z[small] ** 2 / 12)
    result[~small] = x[~small] / np.expm1(z[~small])
    return result


def hh_rate_constants(voltage_mv):
    """Return voltage-dependent HH gate opening and closing rates in 1/ms."""
    alpha_m = 0.1 * _vtrap(-(voltage_mv + 40), 10)
    beta_m = 4 * np.exp(-(voltage_mv + 65) / 18)

    alpha_h = 0.07 * np.exp(-(voltage_mv + 65) / 20)
    beta_h = 1 / (1 + np.exp(-(voltage_mv + 35) / 10))

    alpha_n = 0.01 * _vtrap(-(voltage_mv + 55), 10)
    beta_n = 0.125 * np.exp(-(voltage_mv + 65) / 80)
    return alpha_m, beta_m, alpha_h, beta_h, alpha_n, beta_n


def simulate_hh(stimulus, dt_ms=0.02, return_gates=False):
    """
    Simulate one or more HH neurons with fixed-step forward-Euler integration.

    Parameters
    ----------
    stimulus : array, shape (time,) or (time, conditions)
        Applied current density in µA/cm². Each column is simulated as an
        independent neuron, so many conditions can be run at once.
    dt_ms : float
        Simulation time step in ms.
    return_gates : bool
        If True, also return the gating variables.

    Returns
    -------
    voltage : array with the same shape as stimulus
        Membrane voltage in mV.
    gates : dict of arrays with the same shape as stimulus (only if return_gates=True)
        Keys "m", "h", and "n".
    """
    stimulus = np.asarray(stimulus, dtype=float)
    one_condition = stimulus.ndim == 1
    if one_condition:
        stimulus = stimulus[:, None]

    n_times, n_conditions = stimulus.shape

    # Classic squid-axon HH conductances (mS/cm²) and reversal potentials (mV).
    g_na, e_na = 120.0, 50.0
    g_k, e_k = 36.0, -77.0
    g_leak, e_leak = 0.3, -54.4
    capacitance = 1.0  # µF/cm²

    # Begin at rest and initialize each gate to its steady-state value.
    v = np.full(n_conditions, -65.0)
    am, bm, ah, bh, an, bn = hh_rate_constants(v)
    m = am / (am + bm)
    h = ah / (ah + bh)
    n = an / (an + bn)

    voltage = np.empty((n_times, n_conditions))
    gates = {name: np.empty((n_times, n_conditions)) for name in ("m", "h", "n")}
    voltage[0] = v
    gates["m"][0], gates["h"][0], gates["n"][0] = m, h, n

    for sample in range(n_times - 1):
        # --- Membrane equation ---
        # Ionic currents are positive when positive charge flows OUT of the cell.
        i_na = ...  ## student code: sodium current
        i_k = ...  ## student code: potassium current
        i_leak = ...  ## student code: leak current
        dv_dt = ...  ## student code: solve the membrane equation for dV/dt

        # --- Gate equations ---
        dm_dt = am * (1 - m) - bm * m
        dh_dt = ah * (1 - h) - bh * h
        dn_dt = an * (1 - n) - bn * n

        # --- Forward-Euler update: new value = old value + dt * derivative ---
        v = v + dt_ms * dv_dt
        m = m + dt_ms * dm_dt
        h = h + dt_ms * dh_dt
        n = n + dt_ms * dn_dt

        am, bm, ah, bh, an, bn = hh_rate_constants(v)
        voltage[sample + 1] = v
        gates["m"][sample + 1], gates["h"][sample + 1], gates["n"][sample + 1] = m, h, n

    if one_condition:
        voltage = voltage[:, 0]
        gates = {name: values[:, 0] for name, values in gates.items()}
    return (voltage, gates) if return_gates else voltage


def count_spikes(voltage_mv, threshold_mv=0.0):
    """Count upward crossings of threshold_mv along the time axis."""
    voltage_mv = np.asarray(voltage_mv)
    crossings = (
        (voltage_mv[:-1] < threshold_mv)
        & (voltage_mv[1:] >= threshold_mv)
    )
    return np.sum(crossings, axis=0)


def make_single_pulse(
    amplitude,
    pulse_width_ms,
    onset_ms=20.0,
    duration_ms=60.0,
    dt_ms=0.02,
):
    """Construct one rectangular (monophasic) current pulse. Returns (time_ms, stimulus)."""
    time_ms = np.arange(0, duration_ms + dt_ms, dt_ms)
    stimulus = np.zeros(time_ms.size)
    during_pulse = (
        (time_ms >= onset_ms)
        & (time_ms < onset_ms + pulse_width_ms)
    )
    stimulus[during_pulse] = amplitude
    return time_ms, stimulus

In [ ]:
# Check your membrane equation before moving on. You do not need to edit this cell.
_, check_stimulus = make_single_pulse(amplitude=12.0, pulse_width_ms=1.0)
check_voltage = simulate_hh(
    np.column_stack((np.zeros_like(check_stimulus), check_stimulus))
)
rest_drift_mv = np.max(np.abs(check_voltage[:, 0] + 65.0))

print(f"No stimulus: voltage stays within {rest_drift_mv:.2f} mV of -65 mV (expected: < 0.1 mV).")
print(
    f"12 µA/cm², 1-ms pulse: {count_spikes(check_voltage[:, 1])} spike(s), "
    f"peak {check_voltage[:, 1].max():.1f} mV (expected: 1 spike, peak near +40 mV)."
)

### 2.2 Responses to single pulses

The cell below applies 1-ms pulses of four amplitudes and plots the stimulus, membrane voltage, sodium activation gate $m$, and sodium current $I_{\mathrm{Na}}$. Under our sign convention, inward sodium current is negative; a more negative $I_{\mathrm{Na}}$ therefore means stronger sodium entry into the cell. You do not need to edit this cell.

In [ ]:
# Simulate and plot responses to four single-pulse amplitudes.
dt_ms = 0.02
pulse_width_ms = 1.0
amplitudes = np.array([0.0, 5.0, 7.0, 12.0])

single_pulse_stimuli = []
for amplitude in amplitudes:
    time_ms, stimulus = make_single_pulse(
        amplitude=amplitude, pulse_width_ms=pulse_width_ms, dt_ms=dt_ms
    )
    single_pulse_stimuli.append(stimulus)

single_pulse_stimuli = np.column_stack(single_pulse_stimuli)
single_pulse_voltages, single_pulse_gates = simulate_hh(
    single_pulse_stimuli, dt_ms=dt_ms, return_gates=True
)
single_pulse_i_na = (
    120.0 * single_pulse_gates["m"]**3 * single_pulse_gates["h"]
    * (single_pulse_voltages - 50.0)
)

fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True, height_ratios=(1, 2, 1.3, 1.3))
for condition, amplitude in enumerate(amplitudes):
    axes[0].plot(time_ms, single_pulse_stimuli[:, condition], label=f"{amplitude:g} µA/cm²")
    axes[1].plot(time_ms, single_pulse_voltages[:, condition], label=f"{amplitude:g} µA/cm²")
    axes[2].plot(time_ms, single_pulse_gates["m"][:, condition])
    axes[3].plot(time_ms, single_pulse_i_na[:, condition])

axes[0].set_ylabel("Stimulus\n(µA/cm²)")
axes[1].set_ylabel("Membrane voltage (mV)")
axes[2].set_ylabel("Na activation\ngate m")
axes[3].set(xlabel="Time (ms)", ylabel="$I_{Na}$ (µA/cm²)\n(inward < 0)")
axes[1].axhline(0, color="0.6", linestyle=":", label="spike-count threshold")
axes[3].axhline(0, color="0.6", linewidth=1)
axes[0].legend(ncol=4, fontsize=9)
axes[1].legend(ncol=2, fontsize=9)
fig.suptitle("Responses to 1-ms current pulses")
fig.tight_layout()
plt.show()

**Question 2.1:** Compare the responses as pulse amplitude increases.

**a)** Which tested amplitudes are subthreshold, and which evoke an action potential?

**b)** Does increasing current produce a proportionally larger action-potential peak? Use the $m$ and $I_{\mathrm{Na}}$ traces to explain how the plot illustrates an all-or-none response.

**c)** Why does the 7-µA/cm² pulse produce a later spike than the 12-µA/cm² pulse? (The 1-ms threshold is about 6.9 µA/cm².)



**Answer:**

**a)** 

**b)**  

**c)**  

**Question 2.2:** Why might a current-density value used in this simulation not produce the same neural response if that value were applied through a real electrode in the brain? Name two stimulation-related differences between this simulation and stimulation in the real brain.



**Answer:**




## 3. Effects of stimulation amplitude and frequency

Clinical stimulators commonly deliver **pulse trains** rather than isolated pulses. Pulse-train behavior depends on more than whether one pulse is suprathreshold. At high frequencies, a new pulse may arrive while sodium channels remain inactivated (low $h$) and potassium conductance remains elevated (high $n$) following the previous spike.

We will quantify **pulse following**:

$$
\text{pulse-following ratio}
=
\frac{\text{number of spikes}}{\text{number of delivered pulses}}.
$$

A value of 1 means that the neuron produced one spike per pulse. A value below 1 means that some pulses were not followed by spikes.

### 3.1 Build a pulse train

Complete `make_pulse_train`. A train at `frequency_hz` repeats a pulse every period $T = 1/f$. Because the simulation uses milliseconds, convert the period to ms. Then use `np.arange` to list every pulse onset time from `start_ms` up to (but not including) `stop_ms`, spaced by one period. The provided loop places a rectangular pulse at each onset.

In [ ]:
# TODO: Complete the pulse-train waveform generator.
def make_pulse_train(
    amplitude,
    pulse_width_ms,
    frequency_hz,
    start_ms=50.0,
    stop_ms=450.0,
    duration_ms=500.0,
    dt_ms=0.02,
):
    """
    Construct a train of rectangular depolarizing current pulses.

    Returns (time_ms, stimulus, number_of_pulses).
    """
    time_ms = np.arange(0, duration_ms + dt_ms, dt_ms)
    stimulus = np.zeros(time_ms.size)

    period_ms = ...  ## student code: convert stimulation frequency in Hz to the period in ms
    pulse_onsets_ms = ...  ## student code: every pulse onset from start_ms up to stop_ms

    for onset_ms in pulse_onsets_ms:
        during_pulse = (
            (time_ms >= onset_ms)
            & (time_ms < onset_ms + pulse_width_ms)
        )
        stimulus[during_pulse] = amplitude

    return time_ms, stimulus, len(pulse_onsets_ms)

### 3.2 Sweep amplitude and frequency

The cell below builds one 400-ms pulse train for every combination of five amplitudes and six frequencies, simulates all 30 conditions at once, and counts spikes. Complete the single line that computes the pulse-following ratio for every condition (the arrays `train_spike_counts` and `delivered_pulses` line up element by element).

In [ ]:
# TODO: Sweep pulse amplitude and frequency, then calculate pulse following.
train_amplitudes = np.array([5.0, 7.0, 10.0, 15.0, 25.0])
train_frequencies_hz = np.array([5, 10, 20, 50, 100, 200])
train_pulse_width_ms = 1.0

train_stimuli = []
delivered_pulses = []
for amplitude in train_amplitudes:
    for frequency_hz in train_frequencies_hz:
        time_train_ms, stimulus, n_pulses = make_pulse_train(
            amplitude=amplitude,
            pulse_width_ms=train_pulse_width_ms,
            frequency_hz=frequency_hz,
            dt_ms=dt_ms,
        )
        train_stimuli.append(stimulus)
        delivered_pulses.append(n_pulses)

train_stimuli = np.column_stack(train_stimuli)
delivered_pulses = np.array(delivered_pulses)
train_voltages = simulate_hh(train_stimuli, dt_ms=dt_ms)
train_spike_counts = count_spikes(train_voltages)

pulse_following = ...  ## student code: spikes per delivered pulse, for every condition
pulse_following_grid = pulse_following.reshape(train_amplitudes.size, train_frequencies_hz.size)

fig, ax = plt.subplots(figsize=(8, 4.5))
image = ax.imshow(
    pulse_following_grid, origin="lower", aspect="auto", vmin=0, vmax=1, cmap="viridis"
)
for row in range(train_amplitudes.size):
    for col in range(train_frequencies_hz.size):
        value = pulse_following_grid[row, col]
        ax.text(col, row, f"{value:.2f}", ha="center", va="center",
                color="white" if value < 0.6 else "black", fontsize=9)
ax.set(
    xticks=np.arange(train_frequencies_hz.size),
    xticklabels=train_frequencies_hz,
    yticks=np.arange(train_amplitudes.size),
    yticklabels=train_amplitudes,
    xlabel="Stimulation frequency (Hz)",
    ylabel="Pulse amplitude (µA/cm²)",
    title="Fraction of delivered pulses followed by a spike",
)
fig.colorbar(image, ax=ax, label="Spikes per delivered pulse")
fig.tight_layout()
plt.show()

To understand the heatmap, the next cell re-simulates three 10-µA/cm² trains (50, 100, and 200 Hz) with `return_gates=True`. Each frequency is shown in a separate column, with aligned rows for the stimulus, voltage, sodium inactivation gate $h$, and potassium activation gate $n$. Remember that smaller $h$ means fewer sodium channels are available to start a new spike, and larger $n$ means more potassium current opposing depolarization. You do not need to edit this cell.

In [ ]:
# Compare 1:1 following, every-other-pulse following, and high-frequency failure.
example_amplitude = 10.0
example_frequencies_hz = (50, 100, 200)

example_stimuli = np.column_stack([
    make_pulse_train(example_amplitude, train_pulse_width_ms, frequency_hz, dt_ms=dt_ms)[1]
    for frequency_hz in example_frequencies_hz
])
example_voltages, example_gates = simulate_hh(example_stimuli, dt_ms=dt_ms, return_gates=True)

fig, axes = plt.subplots(
    4, 3, figsize=(13, 8.5), sharex=True, sharey="row",
    height_ratios=(1, 2, 1.3, 1.3),
)
for condition, frequency_hz in enumerate(example_frequencies_hz):
    color = f"C{condition}"
    axes[0, condition].plot(time_train_ms, example_stimuli[:, condition], color=color)
    axes[1, condition].plot(time_train_ms, example_voltages[:, condition], color=color)
    axes[2, condition].plot(time_train_ms, example_gates["h"][:, condition], color=color)
    axes[3, condition].plot(time_train_ms, example_gates["n"][:, condition], color=color)
    axes[0, condition].set_title(f"{frequency_hz} Hz")
    axes[3, condition].set(xlabel="Time (ms)", xlim=(40, 160))

axes[0, 0].set_ylabel("Stimulus\n(µA/cm²)")
axes[1, 0].set_ylabel("Membrane voltage (mV)")
axes[2, 0].set_ylabel("Na inactivation\ngate h")
axes[3, 0].set_ylabel("K activation\ngate n")
fig.suptitle(f"Pulse trains at {example_amplitude:g} µA/cm²: voltage and gating variables")
fig.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()

**Question 3.1:** Use the heatmap and example traces to explain how the response changes with frequency.

**a)** Why do repeated 5-µA/cm² pulses remain subthreshold, even at 200 Hz?

**b)** At 10 µA/cm², why does the neuron follow every pulse at 50 Hz but only every other pulse at 100 Hz? Refer to the sodium-inactivation gate $h$ and potassium-activation gate $n$.

**c)** At 200 Hz, the neuron fires once and then stops. Why do later pulses fail, and how is this different from the 100-Hz response?



**Answer:**

**a)**  

**b)**  

**c)**  

**Question 3.2:** This model shows sustained failure to follow at high stimulation frequency.

How could this observation support the historical idea that high-frequency DBS suppresses nearby cell bodies? Why is this simulation insufficient to explain the complete effect of DBS?



**Answer:**

 

## 4. Strength–duration curve, rheobase, and chronaxie

Pulse amplitude and pulse width trade off against one another. A short pulse has little time to charge the membrane capacitance and therefore needs a high amplitude. A longer pulse can reach threshold with less current.

A **strength–duration curve** plots the minimum current amplitude that evokes an action potential against pulse width. Two clinically useful summary parameters are:

- **Rheobase:** the threshold current approached for a very long pulse.
- **Chronaxie:** the pulse width at which threshold current equals twice the rheobase.

As discussed in lecture, these values depend on the excitable element being stimulated: axons have short chronaxies and cell bodies have long chronaxies.

### 4.1 Find activation thresholds with a binary search

To find the threshold at each pulse width, we use a **binary search** over amplitude. For each pulse width we keep an interval `[lower, upper]` where `lower` is known *not* to evoke a spike and `upper` is known to evoke one. Each iteration:

1. Tests the midpoint of the interval.
2. If the midpoint evoked a spike, the threshold is at or below the midpoint, so the midpoint becomes the new `upper`.
3. Otherwise the threshold is above the midpoint, so the midpoint becomes the new `lower`.

Each iteration halves the interval. For example, starting from `[0, 100]`, a spike at 50 gives `[0, 50]`, no spike at 25 gives `[25, 50]`, and so on.

`find_activation_thresholds` runs one search per pulse width **in parallel**: `lower`, `upper`, `midpoint`, and `did_spike` are arrays with one entry per pulse width. Complete the three marked lines. Because `did_spike` is a Boolean array, use `np.where(condition, value_if_true, value_if_false)`, which chooses elementwise between two arrays.

The function takes the waveform builder as its first argument (`make_stimulus`), so we can reuse it later for biphasic pulses.

In [ ]:
# TODO: Complete the parallel binary search for activation thresholds.
def find_activation_thresholds(
    make_stimulus,
    pulse_widths_ms,
    upper_amplitude=100.0,
    search_iterations=12,
    dt_ms=0.02,
    stimulus_kwargs=None,
):
    """
    Find the minimum amplitude that evokes a spike at each pulse width.

    Parameters
    ----------
    make_stimulus : function
        Waveform builder called as
        make_stimulus(amplitude=..., pulse_width_ms=..., dt_ms=..., **stimulus_kwargs)
        that returns (time_ms, stimulus). For example, make_single_pulse.
    pulse_widths_ms : array
        Pulse widths to test, in ms.
    upper_amplitude : float or array
        Initial upper bound, either one value or one value per pulse width.
        It must evoke a spike at every pulse width.
    search_iterations : int
        Number of halvings. Final resolution is upper_amplitude / 2**search_iterations.
    stimulus_kwargs : dict, optional
        Extra arguments for make_stimulus (for example, an interphase gap).

    Returns
    -------
    thresholds : array
        Estimated threshold amplitude for each pulse width.
    """
    stimulus_kwargs = stimulus_kwargs or {}
    pulse_widths_ms = np.asarray(pulse_widths_ms, dtype=float)

    def spikes_at(amplitudes):
        """Simulate one pulse per pulse width; return a Boolean array of which ones spiked."""
        stimuli = np.column_stack([
            make_stimulus(amplitude=amplitude, pulse_width_ms=width, dt_ms=dt_ms, **stimulus_kwargs)[1]
            for amplitude, width in zip(amplitudes, pulse_widths_ms)
        ])
        voltages = simulate_hh(stimuli, dt_ms=dt_ms)
        return count_spikes(voltages) > 0

    lower = np.zeros(pulse_widths_ms.size)
    upper = np.broadcast_to(np.asarray(upper_amplitude, dtype=float), pulse_widths_ms.shape).copy()
    if not np.all(spikes_at(upper)):
        raise ValueError("upper_amplitude does not evoke a spike at every pulse width; increase it.")

    for _ in range(search_iterations):
        midpoint = ...  ## student code: midpoint of every search interval
        did_spike = spikes_at(midpoint)
        upper = ...  ## student code: where the midpoint spiked, it becomes the new upper bound
        lower = ...  ## student code: where it did not spike, it becomes the new lower bound

    return upper


pulse_widths_ms = np.array([0.1, 0.2, 0.5, 1, 2, 5, 10, 20])
threshold_amplitudes = find_activation_thresholds(make_single_pulse, pulse_widths_ms, dt_ms=dt_ms)

print("Pulse widths (ms):  ", pulse_widths_ms)
print("Thresholds (µA/cm²):", threshold_amplitudes)

### 4.2 Estimate rheobase and chronaxie

Complete `estimate_rheobase_chronaxie`. We cannot simulate an infinitely long pulse, so estimate the **rheobase** as the average threshold of the two longest tested pulses (the last two entries, since pulse widths are sorted in increasing order).

Then estimate the **chronaxie** by linear interpolation with `np.interp(x, xp, fp)`, which returns the value of `fp` at position `x` along the increasing sequence `xp`. Here, `x` is twice the rheobase, `xp` is the thresholds, and `fp` is the pulse widths. Because thresholds *decrease* as pulse width increases, reverse both arrays (`array[::-1]`) so that `xp` is increasing.

In [ ]:
# TODO: Estimate rheobase and chronaxie from a strength-duration curve.
def estimate_rheobase_chronaxie(pulse_widths_ms, thresholds):
    """Return (rheobase, chronaxie_ms). pulse_widths_ms must be sorted in increasing order."""
    rheobase = ...  ## student code: average threshold of the two longest pulses
    chronaxie_ms = ...  ## student code: pulse width where threshold = 2 × rheobase
    return rheobase, chronaxie_ms


rheobase, chronaxie_ms = estimate_rheobase_chronaxie(pulse_widths_ms, threshold_amplitudes)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(pulse_widths_ms, threshold_amplitudes, "o-", label="Simulated activation threshold")
ax.axhline(rheobase, color="tab:green", linestyle="--", label=f"Rheobase ≈ {rheobase:.2f} µA/cm²")
ax.axhline(2 * rheobase, color="tab:orange", linestyle=":", label="2 × rheobase")
ax.axvline(chronaxie_ms, color="tab:red", linestyle="--", label=f"Chronaxie ≈ {chronaxie_ms:.2f} ms")
ax.set(
    xscale="log",
    xlabel="Pulse width (ms; logarithmic scale)",
    ylabel="Threshold amplitude (µA/cm²)",
    title="Strength–duration curve",
)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

print(f"Estimated rheobase: {rheobase:.2f} µA/cm²")
print(f"Estimated chronaxie: {chronaxie_ms:.2f} ms")

**Question 4.1:** Explain the shape of the strength–duration curve.

**a)** Why do short pulses require greater amplitude?

**b)** Why does the curve approach a plateau for long pulses rather than falling toward zero current?



**Answer:**

**a)**  

**b)**  

**Question 4.2:** Use the estimated rheobase and chronaxie to answer the following.

**a)** Report both values and explain what each means.

**b)** Why is this model's chronaxie much longer than the roughly 0.1–0.2 ms values discussed for mammalian axons? (Hint: the classic HH parameters describe an unmyelinated squid giant axon at 6.3 °C.)

**c)** Mammalian axons generally have shorter chronaxies than cell bodies. Why can a sufficiently short pulse preferentially activate an axon?



**Answer:**

**a)**  

**b)**  

**c)**  


## 5. Monophasic and charge-balanced biphasic stimulation

A **monophasic pulse** sends current in only one direction. A **biphasic pulse** adds a second phase with the opposite polarity, often after a short **interphase gap**.

For a rectangular phase, the charge is

$$
Q_{\mathrm{phase}} = I \times PW.
$$

A symmetric biphasic waveform uses equal and opposite amplitudes and equal phase widths, so its signed net charge is zero:

$$
Q_{\mathrm{net}} = Q_1 + Q_2 = 0.
$$

As discussed in lecture, charge balance is important at the electrode–tissue interface because sustained net charge drives irreversible electrochemical reactions that can damage the electrode and tissue. In this section, we ask what the second phase costs **neurally**. Following Section 2, our biphasic pulse is *depolarizing-first*, corresponding to a clinical *cathodic-first* pulse.

### 5.1 Build a biphasic pulse

Complete `make_biphasic_pulse`. Here `pulse_width_ms` is the width of **each** phase (so we can pass this function to `find_activation_thresholds` later). The first phase is already located for you. The second phase begins after the first phase *and* the interphase gap. Calculate `second_start_ms`, then set the stimulus to `+amplitude` during the first phase and `-amplitude` during the second.

The cell then compares a monophasic and a biphasic pulse of the same amplitude (8 µA/cm²) and phase width (1 ms). Complete the line that computes the signed net charge density of each waveform by integrating current over time: sum each column of `comparison_stimuli` and multiply by `dt_ms` (µA/cm² × ms = nC/cm²). The resulting figure also shows $m$ and $I_{\mathrm{Na}}$ so you can see how the second phase changes sodium activation.

In [ ]:
# TODO: Construct a charge-balanced biphasic pulse and compare it with a monophasic pulse.
def make_biphasic_pulse(
    amplitude,
    pulse_width_ms,
    interphase_gap_ms=0.1,
    onset_ms=20.0,
    duration_ms=60.0,
    dt_ms=0.02,
):
    """
    Construct a symmetric, depolarizing-first biphasic pulse.

    pulse_width_ms is the width of EACH phase. Returns (time_ms, stimulus).
    """
    time_ms = np.arange(0, duration_ms + dt_ms, dt_ms)
    stimulus = np.zeros(time_ms.size)

    first_phase = (time_ms >= onset_ms) & (time_ms < onset_ms + pulse_width_ms)
    second_start_ms = ...  ## student code: start of the second phase
    second_phase = (time_ms >= second_start_ms) & (time_ms < second_start_ms + pulse_width_ms)

    stimulus[first_phase] = ...  ## student code: depolarizing first phase
    stimulus[second_phase] = ...  ## student code: equal and opposite second phase
    return time_ms, stimulus


comparison_amplitude = 8.0
comparison_width_ms = 1.0

comparison_time_ms, monophasic_stimulus = make_single_pulse(
    amplitude=comparison_amplitude, pulse_width_ms=comparison_width_ms, dt_ms=dt_ms
)
_, biphasic_stimulus = make_biphasic_pulse(
    amplitude=comparison_amplitude, pulse_width_ms=comparison_width_ms, interphase_gap_ms=0.1, dt_ms=dt_ms
)
comparison_stimuli = np.column_stack((monophasic_stimulus, biphasic_stimulus))
comparison_voltages, comparison_gates = simulate_hh(
    comparison_stimuli, dt_ms=dt_ms, return_gates=True
)
comparison_i_na = (
    120.0 * comparison_gates["m"]**3 * comparison_gates["h"]
    * (comparison_voltages - 50.0)
)

net_charge_density = ...  ## student code: signed net charge density of each waveform (nC/cm²)

for name, charge, spikes in zip(
    ("Monophasic", "Biphasic"), net_charge_density, count_spikes(comparison_voltages)
):
    print(f"{name:>10}: net charge {charge:6.2f} nC/cm², spikes = {spikes}")

fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True, height_ratios=(1, 2, 1.3, 1.3))
for condition, label in enumerate(("Monophasic", "Biphasic")):
    axes[0].plot(comparison_time_ms, comparison_stimuli[:, condition], label=label)
    axes[1].plot(comparison_time_ms, comparison_voltages[:, condition], label=label)
    axes[2].plot(comparison_time_ms, comparison_gates["m"][:, condition])
    axes[3].plot(comparison_time_ms, comparison_i_na[:, condition])
axes[0].axhline(0, color="0.6", linewidth=1)
axes[0].set_ylabel("Stimulus\n(µA/cm²)")
axes[1].set_ylabel("Membrane voltage (mV)")
axes[2].set_ylabel("Na activation\ngate m")
axes[3].set(xlabel="Time (ms)", ylabel="$I_{Na}$ (µA/cm²)\n(inward < 0)", xlim=(15, 45))
axes[3].axhline(0, color="0.6", linewidth=1)
axes[0].legend()
axes[1].legend()
fig.suptitle(f"Monophasic vs. biphasic: {comparison_amplitude:g} µA/cm², {comparison_width_ms:g}-ms phases")
fig.tight_layout()
plt.show()

**Question 5.1:** At 8 µA/cm², the monophasic pulse evokes a spike but the biphasic pulse does not, even though their first phases are identical. Use the voltage, $m$, and $I_{\mathrm{Na}}$ traces to explain what the second phase does and why it can prevent the spike.



**Answer:**

 


### 5.2 The threshold cost of charge balance

Let's quantify that cost with strength–duration curves. The cell below computes monophasic thresholds at five phase widths. Complete the line inside the loop that computes biphasic thresholds for each interphase gap: call `find_activation_thresholds` with `make_biphasic_pulse` as the waveform builder, pass the gap through `stimulus_kwargs={"interphase_gap_ms": gap}`, and use the provided `biphasic_upper_bounds` as the `upper_amplitude` (biphasic thresholds can be many times the monophasic ones, but very large currents make the simulation numerically unstable, so we set one upper bound per phase width).

In [ ]:
# TODO: Compare monophasic and biphasic strength-duration curves for several interphase gaps.
phase_widths_ms = np.array([0.1, 0.2, 0.5, 1, 2])
interphase_gaps_ms = (0.0, 0.1, 1.0)

monophasic_thresholds = find_activation_thresholds(make_single_pulse, phase_widths_ms, dt_ms=dt_ms)
biphasic_upper_bounds = 10 * monophasic_thresholds  # one search upper bound per phase width

biphasic_thresholds = {}
for gap in interphase_gaps_ms:
    biphasic_thresholds[gap] = ... ## student code: biphasic thresholds for this interphase gap

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(phase_widths_ms, monophasic_thresholds, "ko-", label="Monophasic")
for gap in interphase_gaps_ms:
    axes[0].plot(phase_widths_ms, biphasic_thresholds[gap], "o-", label=f"Biphasic, gap {gap:g} ms")
    axes[1].plot(phase_widths_ms, biphasic_thresholds[gap] / monophasic_thresholds, "o-",
                 label=f"gap {gap:g} ms")
axes[0].set(xscale="log", yscale="log", xlabel="Phase width (ms)", ylabel="Threshold (µA/cm²)",
            title="Strength–duration curves")
axes[0].yaxis.set_major_formatter(ScalarFormatter())
axes[1].axhline(1, color="0.6", linestyle="--")
axes[1].set(xscale="log", xlabel="Phase width (ms)", ylabel="Biphasic / monophasic threshold",
            title="Threshold cost of the second phase")
axes[0].legend(fontsize=9)
axes[1].legend(fontsize=9)
fig.tight_layout()
plt.show()

print("Phase widths (ms):", phase_widths_ms)
print("Monophasic:       ", monophasic_thresholds)
for gap in interphase_gaps_ms:
    print(f"Biphasic gap {gap:g} ms:", biphasic_thresholds[gap])

**Question 5.2:**

**a)** How does the biphasic-to-monophasic threshold ratio change with phase width? Explain the pattern using the timing of the second phase.

**b)** How does increasing the interphase gap affect the threshold? Why might a designer avoid making the gap arbitrarily long?



**Answer:**

**a)**  

**b)**  

### 5.3 Electrode charge density and the Shannon criterion

The simulations above used an effective **membrane current density to model extracellular stimulation**. In practice, neural engineers choose an electrode current amplitude and pulse width. To help evaluate whether that stimulation is safe, they calculate the charge delivered through the electrode during each phase, $Q_{\mathrm{phase}}$, and the **charge density**, $D$, which is the charge per unit electrode area:

$$
Q_{\mathrm{phase}} = I_{\mathrm{electrode}} \cdot PW,
\qquad
D = \frac{Q_{\mathrm{phase}}}{A_{\mathrm{electrode}}}.
$$

The **Shannon criterion** discussed in lecture is a historical empirical boundary derived from macroelectrode data:

$$
k = \log_{10}(D) + \log_{10}(Q),
$$

with $D$ in µC/cm² per phase and $Q$ in µC per phase. We will compare the examples with the commonly used boundary $k=1.85$. Being below this boundary is **not** a guarantee of safety; electrode material, waveform, repetition rate, heating, and other factors still matter.

Consider a 3-mA phase lasting 200 µs, delivered through either a DBS-sized contact (0.06 cm²) or a smaller contact (0.005 cm²). Calculate $Q$, $D$, and $k$ for each contact.


In [ ]:
# TODO: Calculate charge per phase, charge density, and Shannon k for two electrode sizes.
electrode_current_ma = 3.0
electrode_phase_width_us = 200.0
electrode_areas_cm2 = np.array([0.06, 0.005])
shannon_k_limit = 1.85

charge_per_phase_uc = ...  ## student code: charge per phase in µC
charge_density_uc_cm2 = ...  ## student code: charge density for each electrode area
shannon_k = ...  ## student code: Shannon k for each electrode

print(f"Charge per phase: {charge_per_phase_uc:.2f} µC")
for area, density, k in zip(electrode_areas_cm2, charge_density_uc_cm2, shannon_k):
    position = "below" if k <= shannon_k_limit else "ABOVE"
    print(f"Area {area:.3f} cm²: D = {density:6.1f} µC/cm²/phase, "
          f"k = {k:.2f} ({position} the illustrative k = {shannon_k_limit} boundary)")

**Question 5.3:**

**a)** Why does the smaller contact have greater charge density even though both contacts deliver the same waveform? How does each value compare with the illustrative Shannon boundary?

**b)** Propose two ways to move the smaller-contact design below the boundary. State one tradeoff associated with each change.

**c)** Why does falling below the Shannon boundary not guarantee safety?



**Answer:**

**a)**  

**b)**  

**c)**  


## Submission checklist

- Run all completed cells from top to bottom and answer every numbered question.
- Save the notebook.
- Submit the notebook itself to Gradescope.
- Export a PDF version and submit it to Gradescope. **Make sure to assign pages in the PDF submission.**